In [2]:
import numpy as np
import pandas as pd
import re 
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


In [3]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sjyti\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [4]:
#printing the stopwords in english
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

In [5]:
#****************Data preprocessing*****************
#loading the dataset to a pandas dataframe
news_dataset= pd.read_csv('FakeNewsNet.csv')

In [6]:
news_dataset.shape

(23196, 5)

In [7]:
news_dataset.head()

,title,news_url,source_domain,tweet_num,real
0,Kandi Burruss Explodes Over Rape Accusation on...,http://toofab.com/2017/05/08/real-housewives-a...,toofab.com,42,1
1,People's Choice Awards 2018: The best red carp...,https://www.today.com/style/see-people-s-choic...,www.today.com,0,1
2,Sophia Bush Sends Sweet Birthday Message to 'O...,https://www.etonline.com/news/220806_sophia_bu...,www.etonline.com,63,1
3,Colombian singer Maluma sparks rumours of inap...,https://www.dailymail.co.uk/news/article-33655...,www.dailymail.co.uk,20,1
4,Gossip Girl 10 Years Later: How Upper East Sid...,https://www.zerchoo.com/entertainment/gossip-g...,www.zerchoo.com,38,1


In [9]:
#missing values in dataset
news_dataset.isnull().sum()

title              0
news_url         330
source_domain    330
tweet_num          0
real               0
dtype: int64

In [10]:
#replacing the null value with empty string 
news_dataset = news_dataset.fillna('') 

In [12]:
news_dataset['content'] = news_dataset['source_domain']+news_dataset['title']

In [13]:
print(news_dataset['content'])

0        toofab.comKandi Burruss Explodes Over Rape Acc...
1        www.today.comPeople's Choice Awards 2018: The ...
2        www.etonline.comSophia Bush Sends Sweet Birthd...
3        www.dailymail.co.ukColombian singer Maluma spa...
4        www.zerchoo.comGossip Girl 10 Years Later: How...
                               ...                        
23191    www.express.co.ukPippa Middleton wedding: In c...
23192    hollywoodlife.comZayn Malik & Gigi Hadid’s Sho...
23193    www.justjared.comJessica Chastain Recalls the ...
23194    www.intouchweekly.comTristan Thompson Feels "D...
23195    www.billboard.comKelly Clarkson Performs a Med...
Name: content, Length: 23196, dtype: object


In [18]:
#separting the data & label 
X=news_dataset.drop(columns='real',axis=1)
Y=news_dataset['real']


In [19]:
print(X)
print(Y)

                                                   title  \
0      Kandi Burruss Explodes Over Rape Accusation on...   
1      People's Choice Awards 2018: The best red carp...   
2      Sophia Bush Sends Sweet Birthday Message to 'O...   
3      Colombian singer Maluma sparks rumours of inap...   
4      Gossip Girl 10 Years Later: How Upper East Sid...   
...                                                  ...   
23191  Pippa Middleton wedding: In case you missed it...   
23192  Zayn Malik & Gigi Hadid’s Shocking Split: Why ...   
23193  Jessica Chastain Recalls the Moment Her Mother...   
23194  Tristan Thompson Feels "Dumped" After Khloé Ka...   
23195  Kelly Clarkson Performs a Medley of Kendrick L...   

                                                news_url  \
0      http://toofab.com/2017/05/08/real-housewives-a...   
1      https://www.today.com/style/see-people-s-choic...   
2      https://www.etonline.com/news/220806_sophia_bu...   
3      https://www.dailymail.co.uk/news

In [23]:
#stemming
#it is the process of reducing a word to its root word
#example :- actor , actress , acting--> act
port_stem=PorterStemmer()
def stemming(content):
    stemmed_content = re.sub('[^a-zA-Z]',' ',content)
    stemmed_content = stemmed_content.lower()
    stemmed_content = stemmed_content.split()
    stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
    stemmed_content = ' '.join(stemmed_content)
    return stemmed_content


In [24]:
news_dataset['content'] =news_dataset['content'].apply(stemming)

In [28]:
print(news_dataset['content'])

0        toofab comkandi burruss explod rape accus real...
1        www today compeopl choic award best red carpet...
2        www etonlin comsophia bush send sweet birthday...
3        www dailymail co ukcolombian singer maluma spa...
4        www zerchoo comgossip girl year later upper ea...
                               ...                        
23191    www express co ukpippa middleton wed case miss...
23192    hollywoodlif comzayn malik gigi hadid shock sp...
23193    www justjar comjessica chastain recal moment m...
23194    www intouchweekli comtristan thompson feel dum...
23195    www billboard comkelli clarkson perform medley...
Name: content, Length: 23196, dtype: object


In [30]:
#separating the data and label 
X= news_dataset['content'].values
Y= news_dataset['real'].values

In [31]:
print(X)
print(Y)

['toofab comkandi burruss explod rape accus real housew atlanta reunion video'
 'www today compeopl choic award best red carpet look'
 'www etonlin comsophia bush send sweet birthday messag one tree hill co star hilari burton breyton eva'
 ...
 'www justjar comjessica chastain recal moment mother boyfriend slap kick genit'
 'www intouchweekli comtristan thompson feel dump khlo kardashian refus let move la home exclus'
 'www billboard comkelli clarkson perform medley kendrick lamar humbl hit billboard music award']
[1 1 1 ... 1 0 1]


In [32]:
Y.shape

(23196,)

In [34]:
#converting the textual data into numerical data
vectorizer = TfidfVectorizer()
vectorizer.fit(X)
X=vectorizer.transform(X)

In [35]:
  print(X)

  (0, 17558)	0.21182632965506829
  (0, 16280)	0.29282877079168707
  (0, 13902)	0.2530032860807785
  (0, 13611)	0.23319435395794588
  (0, 13560)	0.3002668208547604
  (0, 8412)	0.2599228337199497
  (0, 6678)	0.35283888625772475
  (0, 3760)	0.3992969125266123
  (0, 1798)	0.3850682022082062
  (0, 630)	0.3073311902721236
  (0, 57)	0.25808393887630005
  (1, 18141)	0.09188926307468818
  (1, 16245)	0.3326774472870324
  (1, 13682)	0.34075075978911906
  (1, 9997)	0.30172396322540357
  (1, 4278)	0.4826246477349814
  (1, 2354)	0.38104246959127513
  (1, 2047)	0.3494732054103103
  (1, 1200)	0.3098208273796828
  (1, 836)	0.2783761277576564
  (2, 18141)	0.05333413170996292
  (2, 16414)	0.2983921420290402
  (2, 15695)	0.2294893916415434
  (2, 15324)	0.14721282060332433
  (2, 14540)	0.2599067612707775
  :	:
  (23194, 16123)	0.2596370009045142
  (23194, 13728)	0.30653638336232764
  (23194, 10948)	0.2731206551727222
  (23194, 9797)	0.29995574361896993
  (23194, 9572)	0.25704895655596943
  (23194, 9380)	0.

In [36]:
#splitting the dataset into training and test data
X_train,X_test,Y_train,Y_test= train_test_split(X,Y,test_size=0.2,stratify=Y,random_state=2)

In [37]:
model = LogisticRegression()

In [39]:
#training the model
model.fit(X_train,Y_train)

LogisticRegression()

In [40]:
#evaluation
#accuracy score on the training data

X_train_prediction= model.predict(X_train)
training_data_accuracy= accuracy_score(X_train_prediction,Y_train)



In [41]:
print('Accuracy score of the training data : ',training_data_accuracy)


Accuracy score of the training data :  0.8824638930804053


In [44]:
#evaluation
#accuracy score on the test data

X_test_prediction= model.predict(X_test)
test_data_accuracy= accuracy_score(X_test_prediction,Y_test)



In [46]:
print('Accuracy score of the testing data : ',test_data_accuracy)


Accuracy score of the testing data :  0.853448275862069


In [47]:
#making a predictive system
X_new= X_test[0]
prediction =model.predict(X_new)
print(prediction)

if(prediction[0]==0):
    print('The news is real')
else:
    print('The news is fake')

[1]
The news is fake


In [48]:
print(Y_test[0])

1
